# Predictive Customer Churn Classifier
## Random Forest & XGBoost | SMOTE / Class Weighting | ROC-AUC

This notebook predicts telecom customer churn (`Churn = Yes`). It compares:
- **Random Forest** with SMOTE applied only inside the training pipeline.
- **XGBoost** with `scale_pos_weight` to account for class imbalance.

The preprocessing and resampling are fitted within each training fold to avoid data leakage. The held-out test set is never resampled.

**Dataset:** `Telco-Customer-Churn.csv` (place this CSV in the same folder as the notebook).

## 1. Imports and reproducibility

In [ ]:
# If needed, uncomment and run:
# %pip install pandas numpy matplotlib seaborn scikit-learn imbalanced-learn xgboost

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.metrics import (
    roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay,
    classification_report, accuracy_score, precision_score, recall_score, f1_score
)
from sklearn.ensemble import RandomForestClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")


## 2. Load and inspect data

In [ ]:
df = pd.read_csv("Telco-Customer-Churn.csv")
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df["Churn"].value_counts())


## 3. Cleaning and exploratory analysis
`TotalCharges` contains blank strings in this dataset. Convert it to numeric (blanks become missing) and impute it within the preprocessing pipeline. `customerID` is an identifier, so it is excluded from model features.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print("Missing values per column:")
display(df.isna().sum().sort_values(ascending=False).head(10))

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Churn")
plt.title("Customer churn distribution")
plt.tight_layout()
plt.show()

print("Churn rate:", (df["Churn"].eq("Yes").mean() * 100).round(2), "%")


## 4. Split target and features

In [ ]:
X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_transformer = SkPipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Numeric:", numeric_features)
print("Categorical:", categorical_features)


## 5. Build models
SMOTE is used only in the Random Forest pipeline. XGBoost uses the training-fold class ratio via `scale_pos_weight`. Both strategies are applied inside cross-validation / training pipelines, not to the test data.

In [ ]:
minority_count = int(y_train.sum())
majority_count = int((y_train == 0).sum())
scale_pos_weight = majority_count / minority_count

rf_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor),
    ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
    ("model", RandomForestClassifier(
        n_estimators=400, max_depth=None, min_samples_split=4,
        min_samples_leaf=2, class_weight=None,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])

xgb_pipeline = ImbPipeline(steps=[
    ("preprocess", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500, max_depth=4, learning_rate=0.04,
        subsample=0.85, colsample_bytree=0.85,
        reg_lambda=2, objective="binary:logistic",
        eval_metric="logloss", scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=-1
    ))
])


## 6. Stratified cross-validation (ROC-AUC)
Cross-validation is performed on the training split. Each fold refits preprocessing and any resampling independently.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_results = {}

for name, estimator in [("Random Forest + SMOTE", rf_pipeline),
                        ("XGBoost + class weighting", xgb_pipeline)]:
    scores = cross_val_score(estimator, X_train, y_train, cv=cv,
                             scoring="roc_auc", n_jobs=1)
    cv_results[name] = scores
    print(f"{name}: {scores.round(4)} | mean={scores.mean():.4f} ± {scores.std():.4f}")


## 7. Fit final models and evaluate on held-out test data

In [ ]:
models = {
    "Random Forest + SMOTE": rf_pipeline,
    "XGBoost + class weighting": xgb_pipeline
}
predictions = {}
probabilities = {}
rows = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)
    probabilities[name] = y_prob
    predictions[name] = y_pred
    rows.append({
        "Model": name,
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })
    print("\\n", name)
    print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

results = pd.DataFrame(rows).sort_values("ROC-AUC", ascending=False)
display(results.round(4))


## 8. ROC curves

In [ ]:
plt.figure(figsize=(8, 6))
for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, linewidth=2, label=f"{name} (AUC = {auc:.3f})")
plt.plot([0, 1], [0, 1], "k--", label="No-skill baseline")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curves — customer churn")
plt.legend()
plt.tight_layout()
plt.show()


## 9. Confusion matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(name)
plt.tight_layout()
plt.show()


## 10. Top 10 feature importances
Importances are shown for the best ROC-AUC model. One-hot encoded feature names are retained.

In [ ]:
best_name = results.iloc[0]["Model"]
best_pipeline = models[best_name]
fitted_preprocessor = best_pipeline.named_steps["preprocess"]
fitted_model = best_pipeline.named_steps["model"]

feature_names = fitted_preprocessor.get_feature_names_out()
importances = fitted_model.feature_importances_
importance_df = (pd.DataFrame({"Feature": feature_names, "Importance": importances})
                 .sort_values("Importance", ascending=False)
                 .head(10)
                 .sort_values("Importance"))

plt.figure(figsize=(9, 6))
plt.barh(importance_df["Feature"], importance_df["Importance"])
plt.xlabel("Feature importance")
plt.title(f"Top 10 feature importances — {best_name}")
plt.tight_layout()
plt.show()
display(importance_df.sort_values("Importance", ascending=False))


## 11. Conclusion

- Compare the held-out **ROC-AUC** values in the results table; ROC-AUC above 0.80 is a target, not a guaranteed outcome.
- Review recall and the confusion matrix as well as ROC-AUC, because churn detection is affected by the decision threshold.
- The default threshold is 0.50. It can be adjusted based on business costs (missed churners vs. unnecessary retention offers).
- Feature importance indicates model reliance, not causation.

**Leakage safeguards:** the customer ID is excluded, the test set is not resampled, and preprocessing / SMOTE are fitted inside each training fold.